# Taal Vista Hotel Web Scraping

## Purpose

This notebook collects publicly available information from permitted pages of the official Taal Vista Hotel website.

The workflow begins by checking website access rules and inspecting page structure. Data extraction will proceed gradually, beginning with accommodation information.

## Data Collection Principles

1. Collect only publicly accessible business information.

2. Respect website access rules and technical restrictions.

3. Use reasonable request delays.

4. Do not access the external booking system.

5. Do not collect unnecessary personal information.

6. Preserve source URLs and collection dates.

7. Save extracted information in `data/raw` without analytical cleaning.

In [1]:
from pathlib import Path
from datetime import date
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
BASE_URL = "https://www.taalvistahotel.com"
ROBOTS_URL = f"{BASE_URL}/robots.txt"

HEADERS = {
    "User-Agent": "TaalVistaHotelResearchProject/1.0"
}

robots_response = requests.get(
    ROBOTS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", robots_response.status_code)
print()
print(robots_response.text[:3000])

Status code: 200

User-agent: *
Crawl-Delay: 3



## Website Access Check

The website returned asuccessful response for `robots.txt`.

The published rules apply to all automated user agents and specify a crawl delay of three seconds. No disallowed paths were listed.

This project will therefore wait at least three seconds between page requests and will stop if the website returns an access restriction or rate limit response.

In [3]:
robot_parser = RobotFileParser()
robot_parser.set_url(ROBOTS_URL)
robot_parser.parse(robots_response.text.splitlines())

ROOMS_URL = f"{BASE_URL}/rooms/"

print("Rooms page permitted:", robot_parser.can_fetch(HEADERS["User-Agent"], ROOMS_URL))
print("Required crawl delay:", robot_parser.crawl_delay(HEADERS["User-Agent"]))

Rooms page permitted: True
Required crawl delay: 3


In [6]:
import time
import requests

BASE_URL = "https://www.taalvistahotel.com"
ROOMS_URL = f"{BASE_URL}/rooms/"

HEADERS = {
    "User-Agent": "TaalVistaHotelResearchProject/1.0"
}

time.sleep(3)

rooms_response = requests.get(
    ROOMS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", rooms_response.status_code)
print("Final URL:", rooms_response.url)
print("Content type:", rooms_response.headers.get("Content-Type"))
print("HTML characters:", len(rooms_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/rooms/
Content type: text/html; charset=UTF-8
HTML characters: 999862


## Rooms Page Access Result

The rooms page returned a successful HTTP status code of 200 and HTML content.

The page will first be inspected before extraction. This helps identify the correct HTML elements and prevents the scraper from collecting navigation, footer, script, and unrelated page content.

In [7]:
rooms_soup = BeautifulSoup(
    rooms_response.text,
    "lxml"
)

page_title = rooms_soup.title.get_text(
    strip=True
) if rooms_soup.title else "No title found"

print("Page title:", page_title)

Page title: ROOMS - Taal Vista Hotel


In [8]:
page_headings = []

for heading in rooms_soup.find_all(["h1", "h2", "h3", "h4"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text and heading_text not in page_headings:
        page_headings.append(heading_text)

print("Unique headings found:", len(page_headings))
print()

for heading_text in page_headings[:50]:
    print(heading_text)

Unique headings found: 15

Rooms
DELUXE ROOM
PREMIER QUEEN ROOM
Two-Bedroom Deluxe Suite
Batangas Suite
superior room
deluxe room
ridge room
PREMIER ROOM
ONE-BEDROOM DELUXE SUITE
TAAL SUITE
TAGAYTAY SUITE
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


## Initial Rooms Page Findings

The heading inspection identified possible room and suite names together with unrelated website content.

The extraction will not rely only on heading text. The surrounding HTML structure must be inspected to determine whether each room has a description, wing, bed type, view, capacity, room size, or amenities.

In [9]:
excluded_headings = [
    "Rooms",
    "Privacy Overview",
    "Chinese New Year Offerings at Taal Vista Hotel",
    "Breathe Fresh"
]

room_headings = []

for heading_text in page_headings:
    if heading_text not in excluded_headings:
        room_headings.append(heading_text)

print("Possible room headings:", len(room_headings))
print()

for room_heading in room_headings:
    print(room_heading)

Possible room headings: 11

DELUXE ROOM
PREMIER QUEEN ROOM
Two-Bedroom Deluxe Suite
Batangas Suite
superior room
deluxe room
ridge room
PREMIER ROOM
ONE-BEDROOM DELUXE SUITE
TAAL SUITE
TAGAYTAY SUITE


In [10]:
for heading in rooms_soup.find_all(["h1", "h2", "h3", "h4"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text in room_headings:
        print("Room:", heading_text)
        print("Heading tag:", heading.name)
        print("Heading class:", heading.get("class"))
        print("Parent tag:", heading.parent.name)
        print("Parent class:", heading.parent.get("class"))
        print("=" * 50)

Room: DELUXE ROOM
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c30']
Room: PREMIER QUEEN ROOM
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-5629', 'style-local-115-c35']
Room: Two-Bedroom Deluxe Suite
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c44']
Room: Batangas Suite
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c49']
Room: superior room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c60']
Room: deluxe room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c64']
Room: ridge room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c7

In [11]:
sample_room_names = [
    "DELUXE ROOM",
    "PREMIER QUEEN ROOM"
]

for sample_room_name in sample_room_names:
    sample_heading = rooms_soup.find(
        "h3",
        string=lambda text: text and text.strip() == sample_room_name
    )

    print("ROOM:", sample_room_name)

    for level, ancestor in enumerate(sample_heading.parents):
        if level >= 7:
            break

        ancestor_text = ancestor.get_text(
            " ",
            strip=True
        )

        print()
        print("Ancestor level:", level)
        print("Tag:", ancestor.name)
        print("Class:", ancestor.get("class"))
        print("Text preview:", ancestor_text[:400])

    print()
    print("=" * 70)

ROOM: DELUXE ROOM

Ancestor level: 0
Tag: div
Class: ['h-heading__outer', 'style-1580', 'style-local-115-c30']
Text preview: DELUXE ROOM

Ancestor level: 1
Tag: div
Class: ['h-global-transition-all', 'h-heading', 'style-1580', 'style-local-115-c30', 'position-relative', 'h-element']
Text preview: DELUXE ROOM

Ancestor level: 2
Tag: div
Class: ['w-100', 'h-y-container', 'h-column__content', 'h-column__v-align', 'flex-basis-100', 'align-self-lg-start', 'align-self-md-start', 'align-self-start']
Text preview: DELUXE ROOM Stay in our newly renovated Deluxe Rooms, thoughtfully designed for quiet and intimate accommodation. Located in the Lake Wing, guests may choose between a King, Twin, or Queen bed to suit their preference. Each room
                                      offers views of Aguinaldo Highway or the hotel’s charming courtyard, with easy access to our exclusive gate to Skyranch for added lei

Ancestor level: 3
Tag: div
Class: ['d-flex', 'h-flex-basis', 'h-column__inner', 'h-px-

## Initial Room Record Extraction

The repeated `h-column__content` container holds each room heading and its associated descriptive content.

The first extraction will preserve the room names and descriptions as published. Capitalization, category standardization, and feature extraction will be handled later during data cleaning.

In [12]:
room_records = []

for heading in rooms_soup.find_all("h3"):
    room_name = heading.get_text(" ", strip=True)

    if room_name in room_headings:
        room_container = heading.find_parent(
            "div",
            class_="h-column__content"
        )

        paragraph_texts = []

        if room_container:
            for paragraph in room_container.find_all("p"):
                paragraph_text = paragraph.get_text(" ", strip=True)

                if paragraph_text:
                    paragraph_texts.append(paragraph_text)

        room_description = " ".join(paragraph_texts)

        room_records.append({
            "room_name": room_name,
            "room_description": room_description,
            "source_url": ROOMS_URL,
            "date_collected": date.today().isoformat()
        })

rooms_raw_df = pd.DataFrame(room_records)

print("Rows:", rooms_raw_df.shape[0])
print("Columns:", rooms_raw_df.shape[1])
print()
print("Missing values:")
print(rooms_raw_df.isnull().sum())

display(
    rooms_raw_df[
        ["room_name", "room_description"]
    ]
)

Rows: 11
Columns: 4

Missing values:
room_name           0
room_description    0
source_url          0
date_collected      0
dtype: int64


,room_name,room_description
0,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou..."
1,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...
2,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ..."
3,Batangas Suite,Celebrate life’s milestones or simply unwind i...
4,superior room,Surrounded by the refreshing view of lush gree...
5,deluxe room,"Located on a higher floor, the rooms are taste..."
6,ridge room,Inspired by the rich culture and relaxing outd...
7,PREMIER ROOM,"For a more captivating and serene stay, our Pr..."
8,ONE-BEDROOM DELUXE SUITE,Features a spacious living room and the privac...
9,TAAL SUITE,Features a modern and fresh vibe with floor-to...


## Raw Room Data Quality Check

The initial extraction returned 11 room records with no missing descriptions.

Room names will be checked without considering capitalization. A repeated normalized name will not automatically be treated as a duplicate because the records may represent distinct products in different hotel wings.

In [13]:
rooms_raw_df.insert(
    0,
    "room_record_id",
    range(1, len(rooms_raw_df) + 1)
)

rooms_raw_df["description_length"] = (
    rooms_raw_df["room_description"].str.len()
)

rooms_raw_df["normalized_room_name"] = (
    rooms_raw_df["room_name"]
    .str.strip()
    .str.lower()
)

duplicate_name_check = rooms_raw_df[
    rooms_raw_df.duplicated(
        subset="normalized_room_name",
        keep=False
    )
]

print("Exact duplicate rows:", rooms_raw_df.duplicated().sum())
print(
    "Repeated normalized room names:",
    duplicate_name_check.shape[0]
)
print()

display(
    duplicate_name_check[
        [
            "room_record_id",
            "room_name",
            "description_length",
            "room_description"
        ]
    ]
)

Exact duplicate rows: 0
Repeated normalized room names: 2



,room_record_id,room_name,description_length,room_description
0,1,DELUXE ROOM,789,"Stay in our newly renovated Deluxe Rooms, thou..."
5,6,deluxe room,336,"Located on a higher floor, the rooms are taste..."


## Raw Data Export Decision

The two Deluxe Room records will be preserved because they have different descriptions and may represent distinct room products.

Helper columns created for quality checking will not be included in the raw export. The raw dataset will contain only the record identifier, original room name, original description, source URL, and collection date.

In [14]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

raw_data_folder = project_root / "data" / "raw"
raw_data_folder.mkdir(parents=True, exist_ok=True)

rooms_export_df = rooms_raw_df[
    [
        "room_record_id",
        "room_name",
        "room_description",
        "source_url",
        "date_collected"
    ]
].copy()

rooms_file = raw_data_folder / "taal_vista_rooms_raw.csv"

rooms_export_df.to_csv(
    rooms_file,
    index=False
)

print("File saved:", rooms_file)
print("Rows saved:", rooms_export_df.shape[0])
print("Columns saved:", rooms_export_df.shape[1])
print("File exists:", rooms_file.exists())

File saved: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw/taal_vista_rooms_raw.csv
Rows saved: 11
Columns saved: 5
File exists: True


## Raw CSV Verification

The exported CSV will be loaded again to confirm that it is readable and that its dimensions, columns, missing values, and duplicate records remain correct after export.

In [15]:
rooms_verification_df = pd.read_csv(rooms_file)

print("Loaded shape:", rooms_verification_df.shape)
print()
print("Columns:")
print(rooms_verification_df.columns.tolist())
print()
print("Missing values:")
print(rooms_verification_df.isnull().sum())
print()
print(
    "Exact duplicate records:",
    rooms_verification_df.duplicated().sum()
)

display(rooms_verification_df.head())

Loaded shape: (11, 5)

Columns:
['room_record_id', 'room_name', 'room_description', 'source_url', 'date_collected']

Missing values:
room_record_id      0
room_name           0
room_description    0
source_url          0
date_collected      0
dtype: int64

Exact duplicate records: 0


,room_record_id,room_name,room_description,source_url,date_collected
0,1,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou...",https://www.taalvistahotel.com/rooms/,2026-09-02
1,2,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...,https://www.taalvistahotel.com/rooms/,2026-09-02
2,3,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ...",https://www.taalvistahotel.com/rooms/,2026-09-02
3,4,Batangas Suite,Celebrate life’s milestones or simply unwind i...,https://www.taalvistahotel.com/rooms/,2026-09-02
4,5,superior room,Surrounded by the refreshing view of lush gree...,https://www.taalvistahotel.com/rooms/,2026-09-02
